In [3]:
import sqlite3

In [4]:
DB = 'ticket_prices.db'
with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS prices (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            city TEXT UNIQUE,
            price REAL
        )
    ''')
    conn.commit()


In [5]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?)', (city.lower(), price))
        conn.commit()

In [6]:
prices={
    "london": 100.0,
    "paris": 120.0,
    "new york": 200.0,
    "tokyo": 300.0,
}

for city, price in prices.items():
    set_ticket_price(city, price)

In [7]:
def get_ticket_prices(city):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return result[0] if result else 'DATA NOT AVAILABLE'

In [9]:
get_ticket_prices("New York")

200.0

In [11]:
import os
import json
from dotenv import load_dotenv
load_dotenv()


True

In [12]:
groq_api_key = os.getenv("GROQ_API_KEY")
groq_base_url = os.getenv("GROQ_BASE_URL")

if not groq_api_key or not groq_base_url:
    raise ValueError("GROQ_API_KEY and GROQ_BASE_URL must be set in the environment variables.")
else:
    print("GROQ_API_KEY and GROQ_BASE_URL are set correctly.")

GROQ_API_KEY and GROQ_BASE_URL are set correctly.


In [13]:
from openai import OpenAI
llm=OpenAI(
    api_key=groq_api_key,
    base_url=groq_base_url,
)

In [14]:
price_function = {
    "name": "get_ticket_prices",
    "description": "Get the ticket price for a specific city.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "The name of the city to get the ticket price for."
            }
        },
        "required": ["city"]
    }
}


In [46]:
tools=[{"type": "function", "function": price_function}]

In [47]:
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_prices',
   'description': 'Get the ticket price for a specific city.',
   'parameters': {'type': 'object',
    'properties': {'city': {'type': 'string',
      'description': 'The name of the city to get the ticket price for.'}},
    'required': ['city']}}}]

In [60]:
def handle_tool_calls(message):
    responses=[]
    for tool_call in message.tool_calls:
        if tool_call.function.name=="get_ticket_prices":
            arguments=json.loads(tool_call.function.arguments)
            city=arguments.get('city')
            price_details=get_ticket_prices(city)
            responses.append({
                "role":"tool",
                "content":str(price_details),
                "tool_call_id":tool_call.id
            })
    return responses

In [61]:
system_message = """You are a helpful assissant for an Airline called FlightAI. Give short, courteous answers, no more than 1 sentence. Always be accurate. If you don't know the answer, say so """

In [62]:
def chat(message, history):
    history=[{"role":h["role"], "content":h["content"]} for h in history]

    messages=[{"role":"system", "content":system_message}]+history+[{"role":"user", "content":message}]

    response=llm.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=messages
        )
    return response.choices[0].message.content

In [63]:
import gradio as gr
gr.ChatInterface(fn=chat).launch(share=True)

* Running on local URL:  http://127.0.0.1:7880
* Running on public URL: https://bc201b4d12fff249f7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [64]:
def chat(message, history):
    history=[{"role":h["role"], "content":h["content"]} for h in history]

    messages=[{"role":"system", "content":system_message}]+history+[{"role":"user", "content":message}]

    response=llm.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=messages,
        tools=tools,
        tool_choice="auto"
        )
    
    while response.choices[0].finish_reason=="tool_calls":
        message=response.choices[0].message
        response=handle_tool_calls(message)
        messages.append(message)
        messages.extend(response)
        response=llm.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=messages,
        tools=tools,
        tool_choice="auto"
        )
    return response.choices[0].message.content

In [65]:
gr.ChatInterface(fn=chat).launch(share=True)

* Running on local URL:  http://127.0.0.1:7881
* Running on public URL: https://3c1305642e14ebf092.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [68]:
speech_to_text_base_url="https://api.groq.com/openai/v1"

In [77]:
stt=OpenAI(
    api_key=groq_api_key,
    base_url=speech_to_text_base_url,
)

response=stt.audio.transcriptions.create(
    file=open("Data/sample-2.mp3", "rb"),
    model="whisper-large-v3-turbo",
)
print(response.text)

out=response.text


 that every person here, every decision that you've made today, every decision you've made in your life, you've not really made that decision, but in fact,


In [81]:
text_to_speech_base_url="https://api.groq.com/openai/v1"
tts=OpenAI(
    api_key=groq_api_key,
    base_url=text_to_speech_base_url,
)

response = tts.audio.speech.create(
    model="canopylabs/orpheus-v1-english",
    voice="troy",
    input=out,
    response_format="wav"
)
response.write_to_file("output.wav")

In [86]:
def talker(text):
    response = tts.audio.speech.create(
    model="canopylabs/orpheus-v1-english",
    voice="troy",
    input=text,
    response_format="wav"
    )
    return response.content

In [87]:
def chat(history):
    history=[{"role":h["role"], "content":h["content"]} for h in history]
    messages=[{"role":"system", "content":system_message}]+history

    response=llm.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=messages,
        tools=tools,
        tool_choice="auto"
        )
    cities=[]
    
    while response.choices[0].finish_reason=="tool_calls":
        message=response.choices[0].message
        response=handle_tool_calls(message)
        messages.append(message)
        messages.extend(response)
        response=llm.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=messages,
        tools=tools,
        tool_choice="auto"
        )
    reply= response.choices[0].message.content
    history+=[{"role":"assistant", "content":reply}]

    voice=talker(reply)
    return history, voice



In [88]:
def handle_tool_calls_and_return_cities(message):
    responses=[]
    cities=[]
    for tool_call in message.tool_calls:
        if tool_call.function.name=="get_ticket_prices":
            arguments=json.loads(tool_call.function.arguments)
            city=arguments.get('city')
            cities.append(city)
            price_details=get_ticket_prices(city)
            responses.append({
                "role":"tool",
                "content":str(price_details),
                "tool_call_id":tool_call.id
            })
    return responses, cities

In [89]:
def put_message_in_chatbot(message, history):
    return "", history + [{"role": "user", "content": message}]

with gr.Blocks() as ui:
    with gr.Row():
        chatbot=gr.Chatbot()
    with gr.Row():
        audio_output=gr.Audio(autoplay=True)
    with gr.Row():
        message=gr.Textbox(label="Chat with our AI Assistant")

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, audio_output]
    )

ui.launch(share=True)

* Running on local URL:  http://127.0.0.1:7883
* Running on public URL: https://862ad15b36f2a1dae9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
